In [2]:
import pyspark
from pyspark.sql import SparkSession

In [4]:
def getSparkSession():
    spark = SparkSession.builder.appName('students').getOrCreate()
    return spark
def load_data(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    return df
    
filepath='D:/spark/datasets/students.csv'

spark = getSparkSession()
df = load_data(spark, filepath)

In [12]:
df.show()

# df = df.select('gender', 'race/ethnicity','parental level of education', 'test preparation course', 'math score', 'reading score' )

+------+--------------+---------------------------+-----------------------+----------+-------------+
|gender|race/ethnicity|parental level of education|test preparation course|math score|reading score|
+------+--------------+---------------------------+-----------------------+----------+-------------+
|female|       group B|          bachelor's degree|                   none|        72|           72|
|female|       group C|               some college|              completed|        69|           90|
|female|       group B|            master's degree|                   none|        90|           95|
|  male|       group A|         associate's degree|                   none|        47|           57|
|  male|       group C|               some college|                   none|        76|           78|
|female|       group B|         associate's degree|                   none|        71|           83|
|female|       group B|               some college|              completed|        88|     

In [14]:
# 1. Number of male and female students 

male_no = df.filter(df['gender']=='male').count()
female_no = df.filter(df['gender']=='female').count()

print(f'Male count: {male_no}\nFemale Count: {female_no}')

Male count: 482
Female Count: 518


In [15]:
# 2. List different 'race/ethnicity'

races = df.select('race/ethnicity').distinct()
print(f"Races: {races.show()}")

+--------------+
|race/ethnicity|
+--------------+
|       group B|
|       group C|
|       group D|
|       group A|
|       group E|
+--------------+

Races: None


In [17]:
# 3. What are different 'parental level of education'?

parental = df.select('parental level of education').distinct()
print(f"Parental levels of education: {parental.show()}")

+---------------------------+
|parental level of education|
+---------------------------+
|           some high school|
|         associate's degree|
|                high school|
|          bachelor's degree|
|            master's degree|
|               some college|
+---------------------------+

Parental levels of education: None


In [21]:
# 4. How many female students scored more than 79 marks in math, whose parental level of education is 'high school'?

female1 = df.filter(df['gender']=='female').filter(df['math score']>79).filter(df['parental level of education']=='high school').count()
print(female1)

5


In [30]:
from pyspark.sql.functions import avg

# 5. Check whether average maths score of male or female students are high?
avg_score = df.groupBy('gender').agg(avg('math score'))

avg_score.show()

+------+------------------+
|gender|   avg(math score)|
+------+------------------+
|female|63.633204633204635|
|  male| 68.72821576763485|
+------+------------------+



In [31]:
# 6. What is average reading score of male and female students?
avg_readingscore = df.groupBy('gender').agg(avg('reading score'))

avg_readingscore.show()


+------+------------------+
|gender|avg(reading score)|
+------+------------------+
|female| 72.60810810810811|
|  male| 65.47302904564316|
+------+------------------+



In [52]:
# 7. Whether students score depends upon 'parental level of education'? Justify your answer.

import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

resdf = df.withColumn('total_score', F.col('math score') + F.col('reading score'))
resdf = resdf.groupBy('parental level of education').agg(avg('total_score').alias('totalscore')).orderBy('totalscore', ascending=False)
resdf.show()

+---------------------------+------------------+
|parental level of education|        totalscore|
+---------------------------+------------------+
|            master's degree|145.11864406779662|
|          bachelor's degree|142.38983050847457|
|         associate's degree| 138.8108108108108|
|               some college|136.58849557522123|
|           some high school|130.43575418994413|
|                high school|126.84183673469387|
+---------------------------+------------------+



In [53]:
# 8. List the records where 'test preparation course' is 'none' and scored more than 70 in maths. 

df.filter(df['test preparation course']=='none').filter(df['math score']>70).show()

+------+--------------+---------------------------+-----------------------+----------+-------------+-----------+
|gender|race/ethnicity|parental level of education|test preparation course|math score|reading score|total_score|
+------+--------------+---------------------------+-----------------------+----------+-------------+-----------+
|female|       group B|          bachelor's degree|                   none|        72|           72|        144|
|female|       group B|            master's degree|                   none|        90|           95|        185|
|  male|       group C|               some college|                   none|        76|           78|        154|
|female|       group B|         associate's degree|                   none|        71|           83|        154|
|  male|       group C|                high school|                   none|        88|           89|        177|
|  male|       group A|            master's degree|                   none|        73|          